# Week 81, Day 5: Design a URL shortener

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w081d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import string

ALPHABET = string.digits + string.ascii_lowercase + string.ascii_uppercase
print(len(ALPHABET))
for length in (6, 7):
    print(length, f"{len(ALPHABET) ** length:,}")

n = 125
digits = []
while n:
    n, r = divmod(n, 62)
    digits.append(ALPHABET[r])
print("".join(reversed(digits)))

> **Key idea.** Design starts with functional and non-functional requirements, then estimates (here about 40 writes and 4,000 reads a second, 600 GB a year: read-dominated, so cache). Codes are base-62 encodings: of a hash (collisions), random characters (collision checks, unguessable) or a unique counter (no collisions, guessable unless scrambled). Unique ids across servers come from leased ranges or Snowflake ids (41 bits of milliseconds, 10 of machine, 12 of sequence; refuse if the clock goes backwards). The API: POST /links returns 201; GET /{code} redirects with 302 (301 would be cached forever, breaking expiry and counts), 404 for unknown, 410 for expired. Store links by code (shard by code); cache popular links; enqueue click events and count them off the request path; validate URLs, block loops and malicious sites, reserve route names, and rate-limit creation.

### Exercise: Base 62

With `ALPHABET` (digits, then lower-case, then upper-case letters), write `encode(n)`: the base-62 text for a non-negative integer (`encode(0)` is `"0"`; a negative number raises `ValueError("only non-negative numbers")`). Then `decode(code)`: the number back (an empty code, or one with a character not in the alphabet, raises `ValueError(f"bad code: {code!r}")`). And `padded(n, width)`: the code left-padded with `"0"` to `width` characters.

In [ ]:
import string

ALPHABET = string.digits + string.ascii_lowercase + string.ascii_uppercase

def encode(n):
    ___

def decode(code):
    ___

def padded(n, width):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_base62():\n    \"there and back\"\n    assert encode(0) == \"0\" and encode(61) == \"Z\" and encode(62) == \"10\" and encode(125) == \"21\"\n    assert encode(3_521_614_606_207) == \"ZZZZZZZ\" and decode(\"ZZZZZZZ\") == 62 ** 7 - 1\n    assert all(decode(encode(n)) == n for n in (0, 1, 61, 62, 999_999, 2 ** 40))\n    assert padded(125, 7) == \"0000021\" and decode(\"0000021\") == 125\n    for bad in (\"\", \"a-b\", \"\u00fc\"):\n        try:\n            decode(bad)\n        except ValueError as e:\n            assert str(e) == f\"bad code: {bad!r}\"\n        else:\n            raise AssertionError(bad)\n    try:\n        encode(-1)\n    except ValueError as e:\n        assert str(e) == \"only non-negative numbers\"")

### Exercise: Snowflake ids

Write a class `Snowflake(machine_id, clock, epoch_ms=1700000000000)`: `clock()` returns the time in milliseconds. `machine_id` must be from 0 to 1023 (else `ValueError("machine_id must be 0 to 1023")`). `next_id()` returns `(ms - epoch_ms) << 22 | machine_id << 12 | sequence`, where the sequence counts from 0 within the same millisecond and starts again at 0 in a new one. If more than 4,096 ids are asked for in one millisecond, raise `RuntimeError("sequence exhausted in this millisecond")`; if the clock is earlier than the last millisecond used, raise `RuntimeError("clock moved backwards")`. Then `parts(snowflake_id, epoch_ms=1700000000000)`: `(ms, machine_id, sequence)` taken back out of an id.

In [ ]:
class Snowflake:
    ___

def parts(snowflake_id, epoch_ms=1700000000000):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ids():\n    \"unique, sortable and decodable\"\n    class Clock:\n        t = 1794300000000\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    a, b = Snowflake(7, clock), Snowflake(8, clock)\n    ids = [a.next_id(), a.next_id(), b.next_id()]\n    assert parts(ids[0]) == (1794300000000, 7, 0) and parts(ids[1]) == (1794300000000, 7, 1) and parts(ids[2]) == (1794300000000, 8, 0)\n    assert len(set(ids)) == 3\n    clock.t += 1\n    later = a.next_id()\n    assert parts(later) == (1794300000001, 7, 0) and later > max(ids)\n    for _ in range(4095):\n        a.next_id()\n    try:\n        a.next_id()\n    except RuntimeError as e:\n        assert str(e) == \"sequence exhausted in this millisecond\"\n    else:\n        raise AssertionError(\"4097 in one millisecond\")\n    clock.t -= 5\n    try:\n        a.next_id()\n    except RuntimeError as e:\n        assert str(e) == \"clock moved backwards\"\n    else:\n        raise AssertionError(\"backwards\")\n    try:\n        Snowflake(1024, clock)\n    except ValueError as e:\n        assert str(e) == \"machine_id must be 0 to 1023\"")

### Exercise: Leasing id ranges

Write a class `RangeServer(block)`: a central counter that hands out blocks: `lease()` returns `(start, end)` for the next block of `block` ids (the first is `(1, block)`, then `(block + 1, 2 * block)`, and so on) and counts the leases. Then a class `IdAllocator(server)` for one app server: `next_id()` returns the next id from its current block, leasing a new block from the server only when the current one is used up (or it has none yet).

In [ ]:
class RangeServer:
    ___

class IdAllocator:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ranges():\n    \"one central request per block\"\n    central = RangeServer(1000)\n    a, b = IdAllocator(central), IdAllocator(central)\n    assert [a.next_id() for _ in range(3)] == [1, 2, 3] and b.next_id() == 1001\n    ids = [a.next_id() for _ in range(1000)]\n    assert ids[996] == 1000 and ids[997] == 2001 and central.leases == 3\n    everyone = [a.next_id() for _ in range(500)] + [b.next_id() for _ in range(500)]\n    assert len(set(everyone + ids)) == 2000")

### Exercise: Shorten and redirect

Write a class `Shortener(base_url, next_id)`: `next_id()` returns a new unique integer. `shorten(url, now, alias=None, expires_in=None)` returns the code. Refuse with `ValueError`: a URL that isn't `http` or `https` with a host, or longer than 2048 characters (`"not a web address"`); a URL pointing at the shortener's own host (`"links to this service aren't allowed"`); an alias that doesn't fully match `[A-Za-z0-9_-]{3,30}` (`"aliases are 3 to 30 letters, digits, - or _"`), that is one of the reserved words `api`, `links`, `login`, `admin`, `static` in any case (`"that alias is reserved"`), or that's taken (`"that alias is taken"`). Without an alias, the code is the base-62 encoding of `next_id()`. Store the URL with its expiry (`now + expires_in`, or `None`). `resolve(code, now)` returns `(302, url)`; `(404, None)` for an unknown code; `(410, None)` once `now` reaches the expiry. Use the given `encode`.

In [ ]:
import string
from urllib.parse import urlsplit

ALPHABET = string.digits + string.ascii_lowercase + string.ascii_uppercase

def encode(n):
    digits = []
    while True:
        n, r = divmod(n, 62)
        digits.append(ALPHABET[r])
        if n == 0:
            return "".join(reversed(digits))

class Shortener:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shorten():\n    \"codes, aliases and expiry\"\n    ids = iter([125, 126, 3_521_614_606_207])\n    s = Shortener(\"https://sho.rt\", lambda: next(ids))\n    assert s.shorten(\"https://example.org/a/very/long/path?x=1\", 0) == \"21\"\n    assert s.resolve(\"21\", 5) == (302, \"https://example.org/a/very/long/path?x=1\")\n    assert s.shorten(\"https://example.org/party\", 0, alias=\"team-party\", expires_in=60) == \"team-party\"\n    assert s.resolve(\"team-party\", 59) == (302, \"https://example.org/party\") and s.resolve(\"team-party\", 60) == (410, None)\n    assert s.resolve(\"nope\", 0) == (404, None) and s.shorten(\"http://example.com\", 0) == \"22\"\n\ndef test_refusals():\n    \"abuse and mistakes\"\n    s = Shortener(\"https://sho.rt\", iter(range(1, 100)).__next__)\n    s.shorten(\"https://example.org\", 0, alias=\"taken\")\n    for args, message in [((\"javascript:alert(1)\", 0), \"not a web address\"), ((\"https://\", 0), \"not a web address\"),\n                          ((\"https://x.org/\" + \"a\" * 2048, 0), \"not a web address\"),\n                          ((\"https://sho.rt/abc\", 0), \"links to this service aren't allowed\"),\n                          ((\"https://x.org\", 0, \"a b\"), \"aliases are 3 to 30 letters, digits, - or _\"),\n                          ((\"https://x.org\", 0, \"ab\"), \"aliases are 3 to 30 letters, digits, - or _\"),\n                          ((\"https://x.org\", 0, \"Admin\"), \"that alias is reserved\"), ((\"https://x.org\", 0, \"taken\"), \"that alias is taken\")]:\n        try:\n            s.shorten(*args)\n        except ValueError as e:\n            assert str(e) == message, args\n        else:\n            raise AssertionError(args)")

### Exercise: Counting clicks off the request path

Write a class `ClickCounter`. `record(code, timestamp)` only appends `(code, timestamp)` to `self.queue` (this is what the redirect does, and it must stay cheap). `process(batch=1000)` takes up to `batch` events from the front of the queue and adds them to daily counts (the day is `timestamp // 86400`), returning how many it processed. `daily(code)` returns a dict from day number to count for that code, sorted by day. `top(n)` returns the `n` codes with the most clicks processed so far, as `(code, clicks)`, most first, ties by code.

In [ ]:
from collections import Counter, deque

class ClickCounter:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_clicks():\n    \"cheap to record, counted later\"\n    c = ClickCounter()\n    for code, t in [(\"21\", 10), (\"21\", 86_400 + 5), (\"team\", 20), (\"21\", 86_400 + 9), (\"x\", 3), (\"team\", 30)]:\n        c.record(code, t)\n    assert len(c.queue) == 6 and c.top(3) == []\n    assert c.process(batch=4) == 4 and c.daily(\"21\") == {0: 1, 1: 2} and c.daily(\"team\") == {0: 1}\n    assert c.process() == 2 and c.process() == 0\n    assert c.top(2) == [(\"21\", 3), (\"team\", 2)] and c.top(5)[-1] == (\"x\", 1) and c.daily(\"none\") == {}")

### Exercise: A token bucket

Rate-limit link creation with a **token bucket**: each user has a bucket holding up to `capacity` tokens, refilled at `rate` tokens per second (continuously, so after half a second at rate 2 there's one more token); each request takes one token, or is refused if there isn't a whole one. Write a class `TokenBucket(capacity, rate, clock)` with `allow(user)`: `True` (taking a token) or `False`. A new user's bucket starts full. Then `retry_after(user)`: the whole number of seconds (rounded up) until that user will have a token, or `0` if they have one now.

In [ ]:
import math

class TokenBucket:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bucket():\n    \"bursts allowed, rates enforced\"\n    class Clock:\n        t = 0.0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    b = TokenBucket(capacity=3, rate=0.5, clock=clock)\n    assert [b.allow(\"ada\") for _ in range(4)] == [True, True, True, False]\n    assert b.allow(\"bob\") is True and b.retry_after(\"ada\") == 2\n    clock.t = 1.0\n    assert b.allow(\"ada\") is False and b.retry_after(\"ada\") == 1\n    clock.t = 2.0\n    assert b.allow(\"ada\") is True and b.allow(\"ada\") is False\n    clock.t = 100.0\n    assert [b.allow(\"ada\") for _ in range(4)] == [True, True, True, False] and b.retry_after(\"bob\") == 0")

### Bug hunt: Codes read backwards

The decoder reads the code from the wrong end, so `"21"` comes back as `63` instead of `125`, and this prints `125 -> 21 -> 63`. Read the most significant character first, so it prints `125 -> 21 -> 125`.

In [ ]:
import string

ALPHABET = string.digits + string.ascii_lowercase + string.ascii_uppercase

def encode(n):
    out = ""
    while True:
        n, r = divmod(n, 62)
        out = ALPHABET[r] + out
        if n == 0:
            return out

def decode(code):
    n = 0
    for c in reversed(code):
        n = n * 62 + ALPHABET.index(c)
    return n

print(125, "->", encode(125), "->", decode(encode(125)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"most significant first\"\n    expect_output(\"125 -> 21 -> 125\")")

### Bug hunt: A link that hijacked another

Codes are made from a hash of the URL cut down to two characters, and two URLs here happen to share a code. The store doesn't check for a collision, so the second link silently **overwrites** the first, and the first code now sends people to the wrong place: this prints `first link: https://two.example, second link: https://two.example`. If the code is already taken by a **different** URL, try again with the next salt, so the first link keeps its destination: `first link: https://one.example, second link: https://two.example`.

In [ ]:
import hashlib

COLLIDING = {"https://one.example", "https://two.example"}   # their unsalted codes clash, for this demonstration
links = {}

def short_code(url, salt=0):
    if salt == 0 and url in COLLIDING:
        return "ab"
    return hashlib.sha256(f"{url}{salt}".encode()).hexdigest()[:2]

def shorten(url):
    code = short_code(url)
    links[code] = url
    return code

first = shorten("https://one.example")
second = shorten("https://two.example")
print(f"first link: {links[first]}, second link: {links[second]}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"check for collisions\"\n    expect_output(\"first link: https://one.example, second link: https://two.example\")")

### Bug hunt: One extra second of life

The link should stop working at the moment it expires, but the check uses `>`, so at exactly the expiry time it still redirects: this prints `at 59: 302, at 60: 302, at 61: 410`. Make it expire at the expiry time, so it prints `at 59: 302, at 60: 410, at 61: 410`.

In [ ]:
expires_at = 60

def status(now):
    return 410 if now > expires_at else 302

print(", ".join(f"at {t}: {status(t)}" for t in (59, 60, 61)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"expired at the expiry time\"\n    expect_output(\"at 59: 302, at 60: 410, at 61: 410\")")

### Puzzle: Handling a redirect

Arrange a redirect handler that uses a cache, refuses expired links, and counts the click without slowing the response. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # 62 ** 2
q = ___   # the status a shortener returns for a link that has expired
r = ___   # 301 or 302: the redirect browsers may cache forever
s = ___   # the bits of a Snowflake id given to the machine id
t = ___   # writes per second for 100 million links a month (rounded to the nearest 10)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 3844 and q == 410 and r == 301, \"p, q and r\"\n    assert s == 10 and t == 40, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the shortener service

Write a class `ShortenerService(base_url, machine_id, clock)` that combines today's pieces (the given `encode`, `Snowflake`, `TokenBucket`, `LRUCache` and `ClickCounter`); `clock()` returns seconds (a float). `handle(method, path, body, user=None)` returns `(status, headers, body)`:
- `POST /links` with a dict body `{"url", "alias"?, "expires_in"?}`, by a logged-in `user` (else `401`, body `{"error": "log in first"}`): rate-limited per user with a token bucket of capacity 5 refilling at 1 token a minute (`429`, `{"error": "slow down"}` with a `Retry-After` header holding the seconds as text). Validation errors (as in the `shorten` exercise: same messages) give `422` with `{"error": MESSAGE}`. The code is the alias, or `encode` of a Snowflake id (with the clock in milliseconds: `int(clock() * 1000)`). Answer `201` with `{"code": CODE, "short_url": "BASE/CODE"}`, storing the link with its owner and expiry.
- `GET /CODE`: look in an LRU cache of 1000 links first (store a link there on a miss); `404` for unknown, `410` for expired (both with empty dict bodies), else record a click (with `int(clock())`) and answer `302` with `Location`.
- `GET /links/CODE/stats` by the link's owner: process pending clicks, then `200` with `{"code": CODE, "daily": {...}}` (day numbers as keys); anyone else gets `404`.
- Anything else: `404` with `{"error": "not found"}`.
Also count cache hits and misses in `self.cache.hits` and `self.cache.misses`.

In [ ]:
import math
import re
import string
from collections import Counter, OrderedDict, deque
from urllib.parse import urlsplit

ALPHABET = string.digits + string.ascii_lowercase + string.ascii_uppercase
RESERVED = {"api", "links", "login", "admin", "static"}

def encode(n):
    digits = []
    while True:
        n, r = divmod(n, 62)
        digits.append(ALPHABET[r])
        if n == 0:
            return "".join(reversed(digits))

class Snowflake:
    def __init__(self, machine_id, clock_ms, epoch_ms=1700000000000):
        self.machine_id, self.clock_ms, self.epoch_ms = machine_id, clock_ms, epoch_ms
        self.last_ms, self.sequence = -1, 0
    def next_id(self):
        ms = self.clock_ms()
        if ms < self.last_ms:
            raise RuntimeError("clock moved backwards")
        self.sequence = self.sequence + 1 if ms == self.last_ms else 0
        self.last_ms = ms
        return (ms - self.epoch_ms) << 22 | self.machine_id << 12 | self.sequence

class TokenBucket:
    def __init__(self, capacity, rate, clock):
        self.capacity, self.rate, self.clock, self.buckets = capacity, rate, clock, {}
    def _refill(self, user):
        now = self.clock()
        tokens, last = self.buckets.get(user, (self.capacity, now))
        tokens = min(self.capacity, tokens + (now - last) * self.rate)
        self.buckets[user] = (tokens, now)
        return tokens
    def allow(self, user):
        tokens = self._refill(user)
        if tokens >= 1:
            self.buckets[user] = (tokens - 1, self.clock())
            return True
        return False
    def retry_after(self, user):
        tokens = self._refill(user)
        return 0 if tokens >= 1 else math.ceil((1 - tokens) / self.rate)

class LRUCache:
    def __init__(self, capacity):
        self.capacity, self.data, self.hits, self.misses = capacity, OrderedDict(), 0, 0
    def get(self, key):
        if key in self.data:
            self.hits += 1
            self.data.move_to_end(key)
            return self.data[key]
        self.misses += 1
        return None
    def put(self, key, value):
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)

class ClickCounter:
    def __init__(self):
        self.queue, self.days = deque(), {}
    def record(self, code, timestamp):
        self.queue.append((code, timestamp))
    def process(self):
        while self.queue:
            code, t = self.queue.popleft()
            self.days.setdefault(code, Counter())[t // 86400] += 1
    def daily(self, code):
        return dict(sorted(self.days.get(code, {}).items()))

class ShortenerService:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _service():\n    class Clock:\n        t = 1794300000.0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    return ShortenerService(\"https://sho.rt\", 7, clock), clock\n\ndef test_flow():\n    \"create, redirect, count\"\n    s, clock = _service()\n    status, headers, body = s.handle(\"POST\", \"/links\", {\"url\": \"https://example.org/a\"}, user=\"ada\")\n    assert status == 201 and body[\"short_url\"] == \"https://sho.rt/\" + body[\"code\"] and len(body[\"code\"]) >= 7\n    code = body[\"code\"]\n    for _ in range(3):\n        assert s.handle(\"GET\", \"/\" + code, None) == (302, {\"Location\": \"https://example.org/a\"}, {})\n    assert s.cache.misses == 1 and s.cache.hits == 2\n    clock.t += 86400\n    s.handle(\"GET\", \"/\" + code, None)\n    day = int(1794300000 // 86400)\n    assert s.handle(\"GET\", f\"/links/{code}/stats\", None, user=\"ada\") == (200, {}, {\"code\": code, \"daily\": {day: 3, day + 1: 1}})\n    assert s.handle(\"GET\", f\"/links/{code}/stats\", None, user=\"bob\")[0] == 404\n    assert s.handle(\"GET\", \"/missing\", None) == (404, {}, {})\n\ndef test_aliases_expiry_and_errors():\n    \"the edges\"\n    s, clock = _service()\n    assert s.handle(\"POST\", \"/links\", {\"url\": \"https://x.org\"}, user=None) == (401, {}, {\"error\": \"log in first\"})\n    assert s.handle(\"POST\", \"/links\", {\"url\": \"https://x.org\", \"alias\": \"party\", \"expires_in\": 60}, user=\"ada\")[2][\"code\"] == \"party\"\n    assert s.handle(\"POST\", \"/links\", {\"url\": \"https://y.org\", \"alias\": \"party\"}, user=\"bob\") == (422, {}, {\"error\": \"that alias is taken\"})\n    assert s.handle(\"POST\", \"/links\", {\"url\": \"ftp://x\"}, user=\"bob\") == (422, {}, {\"error\": \"not a web address\"})\n    assert s.handle(\"GET\", \"/party\", None)[0] == 302\n    clock.t += 60\n    assert s.handle(\"GET\", \"/party\", None) == (410, {}, {})\n    assert s.handle(\"DELETE\", \"/party\", None) == (404, {}, {\"error\": \"not found\"})\n\ndef test_rate_limit():\n    \"five, then wait\"\n    s, clock = _service()\n    statuses = [s.handle(\"POST\", \"/links\", {\"url\": f\"https://x.org/{i}\"}, user=\"ada\")[0] for i in range(6)]\n    assert statuses == [201, 201, 201, 201, 201, 429]\n    assert s.handle(\"POST\", \"/links\", {\"url\": \"https://x.org/7\"}, user=\"ada\")[1] == {\"Retry-After\": \"60\"}\n    clock.t += 60\n    assert s.handle(\"POST\", \"/links\", {\"url\": \"https://x.org/8\"}, user=\"ada\")[0] == 201\n    assert s.handle(\"POST\", \"/links\", {\"url\": \"https://x.org/9\"}, user=\"bob\")[0] == 201")

## Warm-up

Sixty-two characters (digits and both cases of letter) make 3.5 trillion seven-character codes. Turning a number into a code is just writing it in base 62, the same way you'd write it in base 10 or base 2 (Week 54): `125 = 2 × 62 + 1`, so it's `"21"`.

## Requirements and estimates

A design discussion starts by pinning down what's being built.

**Functional requirements** (what it does):

- shorten a long URL into a short code; optionally with a **custom alias** (`sho.rt/team-party`) and an **expiry**;
- **redirect** anyone opening the short URL to the long one;
- show the link's owner **click counts** by day.

**Non-functional requirements** (how well):

- redirects are **fast** (p99 under 50 ms) and **highly available**: a broken short link on a printed poster can't be fixed;
- codes are **short** and **unguessable enough** that people can't page through everyone's links;
- links don't change once made; the system mustn't become a tool for spreading malicious links.

**Estimates** (Monday's arithmetic): say 100 million new links a month and a read-to-write ratio of 100 to 1. Writes: `10^8 / (30 × 86,400)`, about **40 a second** (perhaps 200 at peak); redirects about **4,000 a second** (20,000 at peak). Each link record is about 500 bytes, so `10^8 × 12 × 500` bytes is **600 GB a year**. Over ten years, 12 billion links: seven base-62 characters (3.5 trillion) is plenty. So: heavily read-dominated, modest storage, and redirect latency is what matters. That points straight at **caching**.

## Short codes and unique ids

Three ways to make codes:

- **Hash the URL** (take the first 7 characters of a base-62 SHA-256): the same URL always gets the same code, but different URLs can **collide**, so you must check and retry.
- **Random codes**: `secrets.choice` seven times; unguessable, but also needs a collision check on each insert, which gets more likely as the table fills.
- **A unique counter, encoded in base 62**: no collisions ever, short codes, and no lookup before insert. The catch is that counters are **guessable** (code `aZ3k` is followed by `aZ3l`), and making one counter work across many servers is the real problem. A common fix for guessability is to scramble the number with a fixed reversible permutation before encoding, or to mix in a few random characters.

**Unique ids across many servers**, without one central counter that every request must ask:

- **Ranges**: a central service hands each server a **block** of ids (1 to 10,000, then 10,001 to 20,000); the server uses its block locally and asks for another when it runs out. One central request per 10,000 ids.
- **Snowflake ids** (from Twitter): a 64-bit number built from the **time** in milliseconds (41 bits), a **machine id** (10 bits) and a per-millisecond **sequence** (12 bits). Each server makes up to 4,096 ids a millisecond with no coordination at all, and the ids sort by creation time. The cost: clocks must not go backwards, so a generator that sees its clock jump back must refuse (or wait) rather than risk a duplicate.

## The API and the data

The API, following Week 77's conventions:

| Request | Answer |
|---|---|
| `POST /links` with `{"url", "alias"?, "expires_in"?}` | `201` with `{"code", "short_url", "expires_at"}` |
| `GET /{code}` | `302` with `Location: LONG_URL`; `404` if unknown; `410 Gone` if expired |
| `GET /links/{code}/stats` | `200` with clicks by day (owner only) |

**`302` or `301`?** A `301 Moved Permanently` lets browsers cache the redirect forever, so repeat visits never reach the service: cheaper, but expiry, changes and click counts stop working for those visitors. A `302` (or `307`) is temporary, so every click comes back. Most shorteners use `302` for that reason.

**Data model**: one table, `links(code PRIMARY KEY, url, owner_id, created_at, expires_at)`, plus a separate store of click counts. Every redirect is a lookup by primary key, which is ideal for a key-value store or a sharded database (Thursday), sharded by `code`.

**Abuse**: validate URLs (only `http` and `https`, a length limit, no links to the shortener itself, which would make loops), check them against lists of known malicious sites, rate-limit link creation per user and per address, and keep aliases from shadowing your own routes (`api`, `links`, `login`).

## Scaling reads and counting clicks

At 20,000 redirects a second, most of them for a few popular links, an in-memory **cache** in front of the database (Tuesday) answers the vast majority: popular links stay in an LRU cache, and since links rarely change, TTLs can be long (with expiry checked on every hit).

**Counting clicks** must not slow redirects down. The redirect handler only **enqueues** a small click event (code, time, country) and returns; workers consume the queue (Wednesday) and update daily counts in batches. If the counters fall behind for a minute, nobody notices; if redirects slowed down for a minute, everyone would.

## Practice

## Quiz

## Daily challenge

## Before you go

Sketch, on paper, how you'd change today's design if links had to be editable after creation, and which of today's decisions would change (start with 301 versus 302). Tomorrow is the Week 81 **Forge**: you'll design and build a small service under load, using everything from the week.